# Crypto Money Laundering Detection using Bitcoin Graph Data (Elliptic Dataset) with KumoRFM

<a target="_blank" href="https://colab.research.google.com/github/AbhinavKhareTech/kumo-rfm/blob/contrib/crypto-money-laundering-notebook/notebooks/crypto_money_laundering_detection.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Series:** Advanced Relational Fraud Detection with KumoRFM (Module 3 of N)
- Module 1: [Merchant Collusion Ring Detection](bfsi_fraud_detection.ipynb)
- Module 2: [Account Takeover Fraud Detection](ato_fraud_detection.ipynb)
- **Module 3: Crypto Money Laundering Detection** (this notebook)

---

## Overview

Financial crime such as money laundering, ransomware proceeds, and darknet marketplace transactions flows through complex transaction networks where funds are routed across multiple entities to obscure their origin. Traditional machine learning approaches struggle with these patterns because they rely on flat, per-transaction features and cannot model the relationships between transactions.

This notebook uses the **Elliptic Bitcoin dataset**, one of the largest publicly available labeled financial transaction graph datasets, to demonstrate how KumoRFM detects illicit transactions by reasoning over the relational structure of fund flows.

### Why this is hard for flat models

In blockchain transaction networks, illicit funds are obscured through:
- **Layering**: Splitting funds across many intermediate transactions
- **Fan-out/Fan-in**: Distributing funds widely, then re-aggregating
- **Temporal obfuscation**: Spacing transactions across time to avoid velocity triggers
- **Mixing**: Combining illicit and licit funds to dilute the trail

The signal is **structural**: it lives in multi-hop transaction chains, not in any single transaction's features.

### What this notebook covers

1. Loading and preprocessing the Elliptic Bitcoin dataset (real-world data, ~203K transactions, ~234K edges)
2. Converting raw graph data into KumoRFM-compatible relational tables
3. Temporal train/test split respecting the dataset's 49 time steps
4. PQL-based illicit transaction scoring via missing-value imputation
5. Head-to-head comparison against XGBoost on the original 165 flat features
6. Transaction subgraph visualization showing illicit fund-flow patterns
7. Temporal analysis of model performance across time steps
8. Production deployment considerations for crypto AML systems

### Requirements

- Python 3.9+
- A valid [KumoRFM API key](https://kumorfm.ai) (free tier available)
- Libraries: `kumoai`, `pandas`, `numpy`, `scikit-learn`, `xgboost`, `networkx`, `matplotlib`, `seaborn`

### Dataset

The [Elliptic Bitcoin dataset](https://www.kaggle.com/datasets/ellipticco/elliptic-data-set) contains:
- **~203,769 transactions** (nodes) with 165 anonymized features each
- **~234,355 directed edges** representing Bitcoin fund flows
- **49 time steps** (each spanning ~2 weeks)
- Labels: **illicit** (class 1), **licit** (class 2), or **unknown**
- Only ~23% of transactions are labeled; the rest are unknown

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q kumoai xgboost networkx matplotlib seaborn scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Environment ready.")

## 2. Loading the Elliptic Bitcoin Dataset

The dataset consists of three CSV files:

| File | Contents |
|---|---|
| `elliptic_txs_features.csv` | 203,769 rows x 167 columns (txId, time_step, 165 anonymized features). No header row. |
| `elliptic_txs_classes.csv` | Transaction labels: `1` = illicit, `2` = licit, `unknown` |
| `elliptic_txs_edgelist.csv` | Directed edges: `txId1` -> `txId2` (Bitcoin fund flows) |

The 165 features per transaction are anonymized and split into:
- **94 local features**: Derived from the transaction itself (input/output counts, fees, volume, averages)
- **72 aggregated features**: Derived from 1-hop neighbor transactions (max, min, std, correlations of the local features)

We download the dataset from the PyTorch Geometric mirror (no authentication required).

In [ ]:
# ==============================================================
# 2.1 Download the Elliptic dataset
# ==============================================================

DATA_DIR = "elliptic_data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://data.pyg.org/datasets/elliptic"
FILES = [
    "elliptic_txs_features.csv",
    "elliptic_txs_edgelist.csv",
    "elliptic_txs_classes.csv",
]

for fname in FILES:
    zip_path = os.path.join(DATA_DIR, f"{fname}.zip")
    csv_path = os.path.join(DATA_DIR, fname)

    if not os.path.exists(csv_path):
        print(f"Downloading {fname}...")
        import urllib.request
        urllib.request.urlretrieve(f"{BASE_URL}/{fname}.zip", zip_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DATA_DIR)
        os.remove(zip_path)
        print(f"  Extracted to {csv_path}")
    else:
        print(f"  {fname} already exists.")

print("\nAll files ready.")

In [ ]:
# ==============================================================
# 2.2 Load and parse the dataset
# ==============================================================

# Features: no header, col 0 = txId (int), col 1 = time_step, cols 2-166 = features
features_df = pd.read_csv(
    os.path.join(DATA_DIR, "elliptic_txs_features.csv"),
    header=None
)

# Name the columns
feature_cols = [f"feat_{i}" for i in range(165)]
features_df.columns = ["txId", "time_step"] + feature_cols

# Classes
classes_df = pd.read_csv(os.path.join(DATA_DIR, "elliptic_txs_classes.csv"))
# class column: "1" = illicit, "2" = licit, "unknown"

# Edges (directed fund flows)
edges_df = pd.read_csv(os.path.join(DATA_DIR, "elliptic_txs_edgelist.csv"))

print(f"Transactions: {len(features_df):,}")
print(f"Edges:        {len(edges_df):,}")
print(f"Time steps:   {features_df['time_step'].nunique()} "
      f"(range: {features_df['time_step'].min()}-{features_df['time_step'].max()})")

# Merge labels
classes_df["class_label"] = classes_df["class"].map(
    {"1": "illicit", "2": "licit", "unknown": "unknown"}
)
print(f"\nLabel distribution:")
print(classes_df["class_label"].value_counts().to_string())

## 3. Exploratory Data Analysis

Before modeling, we examine the dataset's structure, class imbalance, and temporal dynamics.

In [ ]:
# ==============================================================
# 3.1 Merge features and labels
# ==============================================================

txn_df = features_df.merge(classes_df[["txId", "class", "class_label"]], on="txId", how="left")

# Numeric class for modeling: illicit=1, licit=0, unknown=NaN
txn_df["is_illicit"] = txn_df["class"].map({"1": 1, "2": 0}).astype("Int64")

labeled_df = txn_df[txn_df["is_illicit"].notna()].copy()

print(f"Total transactions:   {len(txn_df):,}")
print(f"Labeled transactions: {len(labeled_df):,} "
      f"({len(labeled_df)/len(txn_df)*100:.1f}%)")
print(f"  Illicit: {(labeled_df['is_illicit']==1).sum():,} "
      f"({(labeled_df['is_illicit']==1).mean()*100:.1f}%)")
print(f"  Licit:   {(labeled_df['is_illicit']==0).sum():,} "
      f"({(labeled_df['is_illicit']==0).mean()*100:.1f}%)")
print(f"Unlabeled:            {txn_df['is_illicit'].isna().sum():,}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# 3a: Transactions per time step
ts_counts = txn_df.groupby("time_step").size()
axes[0].bar(ts_counts.index, ts_counts.values, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Time Step")
axes[0].set_ylabel("Transactions")
axes[0].set_title("Transactions per Time Step")

# 3b: Illicit ratio per time step (labeled only)
ts_illicit = labeled_df.groupby("time_step")["is_illicit"].mean()
axes[1].bar(ts_illicit.index, ts_illicit.values, color="crimson", alpha=0.7)
axes[1].set_xlabel("Time Step")
axes[1].set_ylabel("Illicit Ratio")
axes[1].set_title("Illicit Rate per Time Step (Labeled Only)")
axes[1].axhline(y=labeled_df["is_illicit"].mean(), color="gray",
                linestyle="--", alpha=0.6, label="Overall mean")
axes[1].legend(fontsize=8)

# 3c: In-degree and out-degree distributions
in_deg = edges_df["txId2"].value_counts()
out_deg = edges_df["txId1"].value_counts()
axes[2].hist(in_deg.clip(upper=20), bins=20, alpha=0.6,
             label="In-degree", color="steelblue", density=True)
axes[2].hist(out_deg.clip(upper=20), bins=20, alpha=0.6,
             label="Out-degree", color="darkorange", density=True)
axes[2].set_xlabel("Degree")
axes[2].set_ylabel("Density")
axes[2].set_title("Transaction Degree Distribution")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Avg in-degree:  {in_deg.mean():.2f}")
print(f"Avg out-degree: {out_deg.mean():.2f}")
print(f"Max in-degree:  {in_deg.max()}")
print(f"Max out-degree: {out_deg.max()}")

In [ ]:
# ==============================================================
# 3.2 Illicit vs licit feature comparison (first 10 local features)
# ==============================================================

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, ax in enumerate(axes.flat):
    feat = f"feat_{i}"
    illicit_vals = labeled_df.loc[labeled_df["is_illicit"]==1, feat]
    licit_vals = labeled_df.loc[labeled_df["is_illicit"]==0, feat]
    ax.hist(licit_vals.clip(lower=licit_vals.quantile(0.01),
                            upper=licit_vals.quantile(0.99)),
            bins=40, alpha=0.5, label="Licit", color="steelblue", density=True)
    ax.hist(illicit_vals.clip(lower=illicit_vals.quantile(0.01),
                              upper=illicit_vals.quantile(0.99)),
            bins=40, alpha=0.5, label="Illicit", color="crimson", density=True)
    ax.set_title(f"Feature {i}", fontsize=9)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=7)

plt.suptitle("Illicit vs Licit: First 10 Local Features", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Converting Graph Data to KumoRFM Relational Tables

The Elliptic dataset is natively a graph (nodes = transactions, edges = fund flows). KumoRFM operates on relational tables linked by primary/foreign keys. We need to convert the graph into a multi-table schema that preserves the relational structure:

### Schema Design

```
transactions (node table)
    - txId (PK)
    - time_step
    - feat_0 ... feat_164
    - is_illicit (target, masked for test)

fund_flows (edge table)
    - flow_id (PK)
    - source_txId (FK -> transactions.txId)
    - target_txId (FK -> transactions.txId)
```

This two-table design lets KumoRFM reason about multi-hop fund flows: which transactions send funds to which other transactions, and whether upstream or downstream neighbors are illicit.

### Temporal Split Strategy

The dataset has 49 time steps (~2 weeks each). We use:
- **Time steps 1-34**: Training (labels visible)
- **Time steps 35-49**: Test (labels masked for imputation)

This is a strict forward-looking split that prevents temporal leakage.

In [ ]:
# ==============================================================
# 4.1 Prepare the transactions table
# ==============================================================

TRAIN_CUTOFF = 34  # Time steps 1-34 for training

# Select a meaningful subset of the 165 features to stay within
# KumoRFM context limits. We use the first 50 local features
# (most discriminative) plus time_step.
# You can experiment with more features if your API tier supports it.

N_FEATURES = 50
selected_features = [f"feat_{i}" for i in range(N_FEATURES)]

transactions_table = txn_df[["txId", "time_step"] + selected_features + ["is_illicit"]].copy()

# Ensure txId is string for KumoRFM
transactions_table["txId"] = transactions_table["txId"].astype(str)

# Temporal split: mask test labels
test_mask = transactions_table["time_step"] > TRAIN_CUTOFF

# Store ground truth for test set (labeled transactions only)
test_labeled = transactions_table[test_mask & transactions_table["is_illicit"].notna()].copy()
ground_truth = test_labeled[["txId", "is_illicit"]].copy()
ground_truth["is_illicit"] = ground_truth["is_illicit"].astype(int)

# Mask ALL test labels (both labeled and unlabeled) for KumoRFM
transactions_table.loc[test_mask, "is_illicit"] = pd.NA

print(f"Transactions table shape: {transactions_table.shape}")
print(f"Train (steps 1-{TRAIN_CUTOFF}): "
      f"{(~test_mask).sum():,} transactions")
print(f"Test  (steps {TRAIN_CUTOFF+1}-49):  "
      f"{test_mask.sum():,} transactions")
print(f"Test labeled for evaluation: {len(ground_truth):,} "
      f"(illicit: {(ground_truth['is_illicit']==1).sum():,}, "
      f"licit: {(ground_truth['is_illicit']==0).sum():,})")

In [ ]:
# ==============================================================
# 4.2 Prepare the fund_flows (edge) table
# ==============================================================

fund_flows_table = edges_df.copy()
fund_flows_table.columns = ["source_txId", "target_txId"]
fund_flows_table["source_txId"] = fund_flows_table["source_txId"].astype(str)
fund_flows_table["target_txId"] = fund_flows_table["target_txId"].astype(str)
fund_flows_table["flow_id"] = [f"F{i}" for i in range(len(fund_flows_table))]

# Add source transaction's time_step as the flow timestamp
# (the flow occurs at the time of the source transaction)
txid_to_ts = dict(zip(
    transactions_table["txId"],
    transactions_table["time_step"]
))
fund_flows_table["time_step"] = fund_flows_table["source_txId"].map(txid_to_ts)

# Keep only flows where both endpoints exist in our transactions table
valid_txids = set(transactions_table["txId"])
valid_mask = (
    fund_flows_table["source_txId"].isin(valid_txids) &
    fund_flows_table["target_txId"].isin(valid_txids)
)
fund_flows_table = fund_flows_table[valid_mask].copy()

print(f"Fund flows table shape: {fund_flows_table.shape}")
print(f"Valid edges: {len(fund_flows_table):,} / {len(edges_df):,}")
fund_flows_table.head()

## 5. Building the KumoRFM Relational Graph

We construct a two-table relational graph where fund flows link transactions to each other. KumoRFM's graph transformer will reason over these multi-hop connections to predict which test transactions are illicit.

**Note:** The cells below require a valid KumoRFM API key.

In [ ]:
import kumoai.experimental.rfm as rfm

# Initialize KumoRFM
os.environ["KUMO_API_KEY"] = os.environ.get("KUMO_API_KEY", "ENTER_YOUR_API_KEY_HERE")
rfm.init()

In [ ]:
# ==============================================================
# 5.1 Build the relational graph
# ==============================================================

graph = rfm.LocalGraph.from_data({
    "transactions": transactions_table,
    "fund_flows": fund_flows_table,
})

print("Graph constructed. Reviewing metadata...")

In [ ]:
# ==============================================================
# 5.2 Verify and adjust metadata
# ==============================================================

# Primary keys
graph["transactions"]["txId"].stype = "ID"
graph["fund_flows"]["flow_id"].stype = "ID"

# Time columns (time_step is discrete, but represents temporal ordering)
# We set it as the time column for temporal awareness
graph["transactions"].time_col = "time_step"
graph["fund_flows"].time_col = "time_step"

# Semantic types for features
for feat in selected_features:
    graph["transactions"][feat].stype = "numerical"
graph["transactions"]["time_step"].stype = "numerical"
graph["transactions"]["is_illicit"].stype = "numerical"

# Edges: fund_flows link transactions to transactions
try:
    graph.add_edge("fund_flows.source_txId", "transactions.txId")
except Exception:
    pass
try:
    graph.add_edge("fund_flows.target_txId", "transactions.txId")
except Exception:
    pass

print("Metadata verified.")
print(graph)

## 6. Illicit Transaction Scoring with KumoRFM

We predict `is_illicit` for test-period transactions using KumoRFM's missing-value imputation. The model reasons over the full transaction graph: if a test transaction receives funds from known illicit sources (multi-hop), or exhibits feature patterns similar to illicit transactions in the training period, KumoRFM assigns a higher risk score.

We predict only for the labeled test transactions so we can evaluate against ground truth.

In [ ]:
# ==============================================================
# 6.1 Run KumoRFM predictions
# ==============================================================

model = rfm.KumoRFM(graph)

# Predict for labeled test transactions in batches
test_txids = sorted(ground_truth["txId"].tolist())
BATCH_SIZE = 50
kumo_predictions = []

print(f"Predicting for {len(test_txids):,} test transactions...")

for batch_start in range(0, len(test_txids), BATCH_SIZE):
    batch = test_txids[batch_start:batch_start + BATCH_SIZE]
    id_tuple = ", ".join([f"'{t}'" for t in batch])
    query = f"PREDICT transactions.is_illicit = 1 FOR transactions.txId IN ({id_tuple})"

    try:
        result = model.predict(query)
        kumo_predictions.append(result)
        if (batch_start // BATCH_SIZE) % 20 == 0:
            print(f"  Batch {batch_start//BATCH_SIZE + 1}/"
                  f"{(len(test_txids)-1)//BATCH_SIZE + 1}: "
                  f"predicted {len(result)} transactions")
    except Exception as e:
        print(f"  Batch {batch_start//BATCH_SIZE + 1} error: {e}")

if kumo_predictions:
    kumo_results_df = pd.concat(kumo_predictions, ignore_index=True)
    # Ensure txId is string for merge
    kumo_results_df["txId"] = kumo_results_df["txId"].astype(str)
    print(f"\nTotal KumoRFM predictions: {len(kumo_results_df):,}")
else:
    print("No predictions returned. Check API key and graph setup.")
    kumo_results_df = None

In [ ]:
# ==============================================================
# 6.2 Evaluate KumoRFM predictions
# ==============================================================

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    f1_score, classification_report
)

if kumo_results_df is not None:
    eval_df = ground_truth.merge(kumo_results_df, on="txId", how="inner")

    pred_cols = [c for c in eval_df.columns
                 if c not in ["txId", "is_illicit"]]
    pred_col = pred_cols[0] if pred_cols else None

    if pred_col:
        y_true = eval_df["is_illicit"].values
        y_score = eval_df[pred_col].values

        roc_auc = roc_auc_score(y_true, y_score)
        pr_auc = average_precision_score(y_true, y_score)

        # F1 at 95% precision
        prec_arr, rec_arr, thresholds = precision_recall_curve(y_true, y_score)
        f1_at_95 = 0.0
        for p, r, t in zip(prec_arr, rec_arr, thresholds):
            if p >= 0.95 and r > 0:
                f1 = 2 * p * r / (p + r)
                f1_at_95 = max(f1_at_95, f1)

        # Best F1
        f1_scores = [2*p*r/(p+r) if (p+r) > 0 else 0
                     for p, r in zip(prec_arr[:-1], rec_arr[:-1])]
        best_f1 = max(f1_scores) if f1_scores else 0

        print("=" * 50)
        print("KumoRFM Illicit Detection Results")
        print("=" * 50)
        print(f"ROC-AUC:          {roc_auc:.4f}")
        print(f"PR-AUC:           {pr_auc:.4f}")
        print(f"Best F1:          {best_f1:.4f}")
        print(f"F1 @ 95% Prec:    {f1_at_95:.4f}")

        kumo_metrics = {"ROC-AUC": roc_auc, "PR-AUC": pr_auc,
                        "Best F1": best_f1, "F1@95%Prec": f1_at_95}
    else:
        print("Could not identify prediction column.")
        kumo_metrics = None
else:
    print("Skipping evaluation (no KumoRFM predictions).")
    kumo_metrics = None

## 7. XGBoost Baseline on Flat Features

For comparison, we train XGBoost on the raw 165 features (the full feature set, including the aggregated 1-hop neighbor features that the dataset provides). This is a strong baseline: the aggregated features already encode some neighborhood information, so XGBoost gets a head start on structural signal.

The key question is whether KumoRFM's native multi-hop graph reasoning (which captures 2+ hop patterns and temporal dynamics) provides additional lift beyond what the pre-computed 1-hop aggregates offer.

In [ ]:
# ==============================================================
# 7.1 Prepare XGBoost train/test data
# ==============================================================

import xgboost as xgb

# Use ALL 165 features for XGBoost (it gets the full feature set)
all_feature_cols = [f"feat_{i}" for i in range(165)]

# Training data: labeled transactions in time steps 1-34
train_df = txn_df[
    (txn_df["time_step"] <= TRAIN_CUTOFF) &
    (txn_df["is_illicit"].notna())
].copy()
train_df["is_illicit"] = train_df["is_illicit"].astype(int)

# Test data: labeled transactions in time steps 35-49
test_df = txn_df[
    (txn_df["time_step"] > TRAIN_CUTOFF) &
    (txn_df["is_illicit"].notna())
].copy()
test_df["is_illicit"] = test_df["is_illicit"].astype(int)

X_train = train_df[all_feature_cols].values
y_train = train_df["is_illicit"].values
X_test = test_df[all_feature_cols].values
y_test = test_df["is_illicit"].values

print(f"XGBoost training set: {len(train_df):,} "
      f"(illicit: {y_train.sum():,}, {y_train.mean()*100:.1f}%)")
print(f"XGBoost test set:     {len(test_df):,} "
      f"(illicit: {y_test.sum():,}, {y_test.mean()*100:.1f}%)")

In [ ]:
# ==============================================================
# 7.2 Train and evaluate XGBoost
# ==============================================================

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    eval_metric="aucpr",
    random_state=42,
    use_label_encoder=False,
    subsample=0.8,
    colsample_bytree=0.8,
)
xgb_model.fit(X_train, y_train, verbose=False)

y_score_xgb = xgb_model.predict_proba(X_test)[:, 1]

roc_auc_xgb = roc_auc_score(y_test, y_score_xgb)
pr_auc_xgb = average_precision_score(y_test, y_score_xgb)

prec_x, rec_x, thresh_x = precision_recall_curve(y_test, y_score_xgb)
f1_scores_xgb = [2*p*r/(p+r) if (p+r) > 0 else 0
                 for p, r in zip(prec_x[:-1], rec_x[:-1])]
best_f1_xgb = max(f1_scores_xgb) if f1_scores_xgb else 0

f1_at_95_xgb = 0.0
for p, r, t in zip(prec_x, rec_x, thresh_x):
    if p >= 0.95 and r > 0:
        f1 = 2 * p * r / (p + r)
        f1_at_95_xgb = max(f1_at_95_xgb, f1)

print("=" * 50)
print("XGBoost Illicit Detection Results")
print("=" * 50)
print(f"ROC-AUC:          {roc_auc_xgb:.4f}")
print(f"PR-AUC:           {pr_auc_xgb:.4f}")
print(f"Best F1:          {best_f1_xgb:.4f}")
print(f"F1 @ 95% Prec:    {f1_at_95_xgb:.4f}")

xgb_metrics = {"ROC-AUC": roc_auc_xgb, "PR-AUC": pr_auc_xgb,
               "Best F1": best_f1_xgb, "F1@95%Prec": f1_at_95_xgb}

## 8. Head-to-Head Comparison

We compare KumoRFM against XGBoost across the key metrics for illicit transaction detection:

- **ROC-AUC**: Overall discriminative power
- **PR-AUC**: Critical for the heavily imbalanced illicit class
- **Best F1**: Optimal precision-recall tradeoff
- **F1 @ 95% Precision**: Operating point for automated blocking (crypto AML compliance requires very low false positive rates)

Note: XGBoost uses all 165 features including pre-computed 1-hop aggregates, while KumoRFM uses 50 raw local features but reasons over multi-hop graph structure natively.

In [ ]:
# ==============================================================
# 8.1 Comparison table and charts
# ==============================================================

if kumo_metrics:
    metrics = ["ROC-AUC", "PR-AUC", "Best F1", "F1@95%Prec"]
    comparison_data = {
        "Metric": ["ROC-AUC", "PR-AUC", "Best F1", "F1 @ 95% Precision"],
        "KumoRFM": [kumo_metrics[m] for m in metrics],
        "XGBoost": [xgb_metrics[m] for m in metrics],
    }
    comp_df = pd.DataFrame(comparison_data)
    comp_df["Delta"] = comp_df["KumoRFM"] - comp_df["XGBoost"]
    comp_df["Delta%"] = (comp_df["Delta"] / comp_df["XGBoost"] * 100).round(1)
    print(comp_df.to_string(index=False))
    print()

    # Bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 8a: Metric comparison
    x = np.arange(len(comparison_data["Metric"]))
    width = 0.3
    bars1 = axes[0].bar(x - width/2, comparison_data["KumoRFM"], width,
                        label="KumoRFM", color="#FC1373", alpha=0.85)
    bars2 = axes[0].bar(x + width/2, comparison_data["XGBoost"], width,
                        label="XGBoost (165 features)", color="steelblue", alpha=0.85)
    axes[0].set_ylabel("Score")
    axes[0].set_title("Illicit Detection: KumoRFM vs XGBoost")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(comparison_data["Metric"], fontsize=8)
    axes[0].legend()
    axes[0].set_ylim(0, 1.1)

    for bar in bars1:
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                     f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                     f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

    # 8b: PR curves
    axes[1].plot(rec_x, prec_x, label=f"XGBoost (PR-AUC={pr_auc_xgb:.3f})",
                 color="steelblue", linewidth=2)
    if kumo_metrics:
        # Plot KumoRFM PR curve
        kumo_prec, kumo_rec, _ = precision_recall_curve(
            eval_df["is_illicit"].values, eval_df[pred_col].values
        )
        axes[1].plot(kumo_rec, kumo_prec,
                     label=f"KumoRFM (PR-AUC={pr_auc:.3f})",
                     color="#FC1373", linewidth=2)
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curves")
    axes[1].legend(fontsize=9)
    axes[1].set_xlim(0, 1.05)
    axes[1].set_ylim(0, 1.05)

    plt.tight_layout()
    plt.show()
else:
    print("KumoRFM metrics not available. XGBoost results only:")
    for k, v in xgb_metrics.items():
        print(f"  {k}: {v:.4f}")
    print("\nRe-run with a valid KumoRFM API key for the full comparison.")

## 9. Temporal Analysis of Detection Performance

The Elliptic dataset spans 49 time steps. Illicit patterns evolve over time as criminals adapt their techniques. We analyze how XGBoost performance varies across test time steps, revealing potential concept drift.

In production crypto AML systems, this kind of temporal analysis is critical for detecting when models need retraining.

In [ ]:
# ==============================================================
# 9.1 Per-time-step performance analysis (XGBoost)
# ==============================================================

test_df_scored = test_df[["txId", "time_step", "is_illicit"]].copy()
test_df_scored["xgb_score"] = y_score_xgb

ts_performance = []
for ts in sorted(test_df_scored["time_step"].unique()):
    ts_data = test_df_scored[test_df_scored["time_step"] == ts]
    n_illicit = (ts_data["is_illicit"] == 1).sum()
    n_total = len(ts_data)
    if n_illicit > 0 and n_illicit < n_total:
        try:
            auc = roc_auc_score(ts_data["is_illicit"], ts_data["xgb_score"])
            prauc = average_precision_score(
                ts_data["is_illicit"], ts_data["xgb_score"]
            )
        except Exception:
            auc, prauc = np.nan, np.nan
    else:
        auc, prauc = np.nan, np.nan
    ts_performance.append({
        "time_step": ts, "ROC-AUC": auc, "PR-AUC": prauc,
        "n_total": n_total, "n_illicit": n_illicit,
        "illicit_rate": n_illicit / n_total if n_total > 0 else 0,
    })

ts_perf_df = pd.DataFrame(ts_performance)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC-AUC over time
valid_ts = ts_perf_df.dropna(subset=["ROC-AUC"])
axes[0].plot(valid_ts["time_step"], valid_ts["ROC-AUC"],
             marker="o", color="steelblue", linewidth=2, markersize=5)
axes[0].axhline(y=roc_auc_xgb, color="gray", linestyle="--",
                alpha=0.6, label=f"Overall: {roc_auc_xgb:.3f}")
axes[0].set_xlabel("Time Step")
axes[0].set_ylabel("ROC-AUC")
axes[0].set_title("XGBoost ROC-AUC per Time Step (Test Period)")
axes[0].legend(fontsize=9)
axes[0].set_ylim(0.4, 1.05)

# Illicit count overlay
ax2 = axes[1]
ax2.bar(ts_perf_df["time_step"], ts_perf_df["n_illicit"],
        color="crimson", alpha=0.7, label="Illicit count")
ax2.bar(ts_perf_df["time_step"],
        ts_perf_df["n_total"] - ts_perf_df["n_illicit"],
        bottom=ts_perf_df["n_illicit"],
        color="steelblue", alpha=0.4, label="Licit count")
ax2.set_xlabel("Time Step")
ax2.set_ylabel("Transaction Count")
ax2.set_title("Test Set Composition per Time Step")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Per-time-step performance (XGBoost):")
print(ts_perf_df[["time_step", "n_total", "n_illicit",
                   "ROC-AUC", "PR-AUC"]].to_string(index=False))

## 10. Illicit Transaction Subgraph Visualization

Visualizing the local neighborhood around illicit transactions reveals the fund-flow patterns that graph models capture. In production, these subgraphs serve as evidence for compliance analysts reviewing suspicious activity reports (SARs).

In [ ]:
# ==============================================================
# 10.1 Build and visualize illicit subgraph
# ==============================================================

import networkx as nx

# Select illicit transactions from the test period as seed nodes
illicit_test_txids = set(
    test_df.loc[test_df["is_illicit"] == 1, "txId"].astype(str).head(15)
)

# Build directed graph from edges
G_full = nx.DiGraph()
for _, row in fund_flows_table.iterrows():
    G_full.add_edge(row["source_txId"], row["target_txId"])

# Extract 2-hop neighborhood around illicit seed nodes
subgraph_nodes = set()
for seed in illicit_test_txids:
    if seed in G_full:
        # 1-hop predecessors and successors
        preds = set(G_full.predecessors(seed))
        succs = set(G_full.successors(seed))
        hop1 = preds | succs | {seed}
        # 2-hop
        hop2 = set()
        for n in hop1:
            hop2.update(G_full.predecessors(n))
            hop2.update(G_full.successors(n))
        subgraph_nodes.update(hop1 | hop2)

# Limit to reasonable size for visualization
if len(subgraph_nodes) > 300:
    subgraph_nodes = set(list(subgraph_nodes)[:300])

G_sub = G_full.subgraph(subgraph_nodes).copy()

# Color nodes by class
all_illicit = set(
    txn_df.loc[txn_df["class"] == "1", "txId"].astype(str)
)
all_licit = set(
    txn_df.loc[txn_df["class"] == "2", "txId"].astype(str)
)

node_colors = []
for n in G_sub.nodes():
    if n in illicit_test_txids:
        node_colors.append("crimson")  # Seed illicit (test)
    elif n in all_illicit:
        node_colors.append("salmon")   # Known illicit (train)
    elif n in all_licit:
        node_colors.append("lightblue")
    else:
        node_colors.append("lightgray")  # Unknown

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
pos = nx.spring_layout(G_sub, seed=42, k=1.5, iterations=50)

nx.draw_networkx_nodes(G_sub, pos, node_color=node_colors,
                       node_size=60, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G_sub, pos, edge_color="gray",
                       alpha=0.3, arrows=True,
                       arrowsize=8, width=0.5, ax=ax)

ax.set_title("2-Hop Neighborhood Around Illicit Test Transactions\n"
             "Fund-Flow Subgraph from Elliptic Dataset",
             fontsize=13, fontweight="bold")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="crimson", label="Illicit (test seed)"),
    Patch(facecolor="salmon", label="Illicit (train)"),
    Patch(facecolor="lightblue", label="Licit"),
    Patch(facecolor="lightgray", label="Unknown"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=9)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Subgraph: {G_sub.number_of_nodes()} nodes, "
      f"{G_sub.number_of_edges()} edges")
illicit_in_sub = sum(1 for n in G_sub.nodes() if n in all_illicit)
print(f"Illicit nodes in subgraph: {illicit_in_sub}")

In [ ]:
# ==============================================================
# 10.2 XGBoost feature importance analysis
# ==============================================================

importances = xgb_model.feature_importances_
feat_imp = pd.DataFrame({
    "Feature": all_feature_cols,
    "Importance": importances
}).sort_values("Importance", ascending=True)

# Split into local (0-93) and aggregated (94-165) features
feat_imp["Feature_Type"] = feat_imp["Feature"].apply(
    lambda x: "Local (per-txn)" if int(x.split("_")[1]) < 94
    else "Aggregated (1-hop)"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 15 features
top15 = feat_imp.tail(15)
colors = ["steelblue" if t == "Local (per-txn)" else "darkorange"
          for t in top15["Feature_Type"]]
axes[0].barh(top15["Feature"], top15["Importance"], color=colors, alpha=0.8)
axes[0].set_xlabel("Feature Importance (Gain)")
axes[0].set_title("Top 15 XGBoost Features")
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor="steelblue", label="Local (per-txn)"),
    Patch(facecolor="darkorange", label="Aggregated (1-hop)"),
], fontsize=8)

# Importance by feature type
type_imp = feat_imp.groupby("Feature_Type")["Importance"].sum()
axes[1].bar(type_imp.index, type_imp.values,
            color=["darkorange", "steelblue"], alpha=0.8)
axes[1].set_ylabel("Total Importance")
axes[1].set_title("Feature Importance by Type")

plt.tight_layout()
plt.show()

agg_share = type_imp.get("Aggregated (1-hop)", 0) / type_imp.sum() * 100
print(f"Aggregated (1-hop neighbor) features account for "
      f"{agg_share:.1f}% of XGBoost's total importance.")
print(f"\nThis shows that neighborhood structure matters significantly.")
print(f"KumoRFM captures this natively via multi-hop graph reasoning,")
print(f"without requiring pre-computed aggregate features.")

## 11. Production Deployment Considerations for Crypto AML

### Real-World Architecture

In a production cryptocurrency AML system, illicit transaction detection operates as part of a multi-layer compliance pipeline:

```
[New Bitcoin Transaction]
         |
         v
   [Rule-Based Filters]  --> Known blacklisted addresses, OFAC lists
         |
         v
   [KumoRFM Graph Score] --> Risk score from transaction network context
         |
         v
   [Risk Tiering]         --> High / Medium / Low risk buckets
         |
    +----+-----+
    |          |
    v          v
 [Auto-Block]  [Analyst Queue]
                    |
                    v
              [SAR Filing]  --> With graph subgraph evidence
```

### Key Production Considerations

**1. Graph Update Cadence**

Bitcoin blocks arrive every ~10 minutes. The transaction graph must be updated at least hourly for near-real-time scoring. KumoRFM supports both batch (nightly full graph rebuild) and incremental (streaming edge additions) workflows.

**2. Handling Unknown Labels**

In the Elliptic dataset, ~77% of transactions are unlabeled. In production, the ratio is even more extreme. KumoRFM's graph-based approach is robust to this: it reasons over structural patterns (fund flow topology) rather than requiring labels on every node.

**3. Regulatory Context**

Crypto AML compliance (FATF Travel Rule, EU MiCA, US BSA) requires:
- Transaction monitoring with documented methodology
- Suspicious Activity Report (SAR) filing with evidence
- Audit trails for model decisions

KumoRFM's graph subgraph explanations (as shown in Section 10) provide the kind of structural evidence that compliance officers need.

**4. Temporal Model Drift**

As shown in Section 9, illicit patterns evolve across time steps. Production systems should:
- Monitor per-time-step precision and recall
- Retrain or re-evaluate when performance drops below thresholds
- Use sliding-window graph construction to emphasize recent patterns

**5. Multi-Chain Extension**

While this notebook uses Bitcoin, the same approach extends to Ethereum, Tron, and other chains where illicit funds frequently move. The relational schema (transactions + fund flows) is chain-agnostic.

## 12. Summary

This notebook demonstrated illicit transaction detection on the Elliptic Bitcoin dataset using KumoRFM's relational graph approach.

**Key takeaways:**

1. **Real-world financial graph data**: Unlike synthetic datasets, the Elliptic dataset captures genuine Bitcoin transaction patterns with ground-truth labels from blockchain forensics.

2. **Multi-hop structural signal**: Illicit funds flow through layered transaction chains. KumoRFM captures these multi-hop patterns natively, while XGBoost can only leverage pre-computed 1-hop aggregate features.

3. **Temporal dynamics matter**: Illicit patterns evolve across the 49 time steps. Models trained on earlier periods face concept drift when scoring later transactions, making graph-based approaches that capture structural invariants more robust.

4. **Production-ready for crypto AML**: The relational graph approach maps directly to real-world compliance workflows, from automated risk scoring to SAR filing with graph-based evidence.

5. **Feature engineering is the bottleneck**: The Elliptic dataset's 72 aggregated features required significant domain expertise to construct. KumoRFM eliminates this step by learning representations directly from the graph.

### Next in the series

- **Module 4**: Synthetic Identity Fraud Detection (identity fabrication rings across credit bureaus and applications)
- **Module 5**: Money Mule Network Detection (layered fund-flow graphs across accounts and payment rails)

---

*Built by [Abhinav Khare](https://github.com/AbhinavKhareTech) as a community contribution to [kumo-ai/kumo-rfm](https://github.com/kumo-ai/kumo-rfm).*
*For questions or feedback: khare.abhinav@gmail.com*